<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M05/M05_Lab1_Embeddings_VectorStores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🧮 M05 Lab 1 — Embeddings & Vector Stores</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Create <b>text embeddings</b> using the OpenAI API</li>
    <li>Visualize <b>embedding similarity</b> with a cosine similarity matrix</li>
    <li>Set up a <b>ChromaDB vector store</b> and add documents</li>
    <li>Query by <b>semantic meaning</b> (not just keywords)</li>
    <li>Compare <b>keyword search vs semantic search</b></li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai chromadb matplotlib numpy
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from openai import OpenAI
from dads5250 import setup_openai, pp, show_response, show_info, compare_responses, quiz
import numpy as np
import matplotlib.pyplot as plt

client = setup_openai()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ What Are Embeddings?</h2>
</div>

An **embedding** is a vector (list of numbers) that represents the **meaning** of text. Similar meanings → similar vectors.

```
"cat"   → [0.12, -0.34, 0.56, ...] (1536 dimensions)
"kitten" → [0.11, -0.33, 0.57, ...] ← very similar!
"car"   → [0.89, 0.22, -0.44, ...] ← very different
```

This is the foundation of:
- **Semantic search** (search by meaning, not keywords)
- **RAG** (finding relevant documents for an LLM)
- **Clustering** (grouping similar items)
- **Recommendation** systems

In [ ]:
# ============================================================
# 🧮 Create Embeddings with OpenAI
# ============================================================

# Sample texts — some are semantically similar, some aren't
texts = [
    "The cat sat on the mat",
    "A kitten was resting on the rug",
    "Dogs are loyal companions",
    "The stock market crashed today",
    "Financial markets experienced a sharp decline",
    "I love eating pizza on Fridays",
    "Supply chain disruptions affected global trade",
    "Logistics bottlenecks slowed international commerce",
]

# Get embeddings from OpenAI
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

# Extract the vectors
embeddings = [item.embedding for item in response.data]

print(f"✅ Created {len(embeddings)} embeddings")
print(f"📐 Each embedding has {len(embeddings[0])} dimensions")
print(f"\n📊 First 5 values of 'The cat sat on the mat':")
print(f"   {embeddings[0][:5]}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Cosine Similarity Matrix</h2>
</div>

**Cosine similarity** measures how similar two vectors are — a score of 1.0 means identical, 0.0 means unrelated.

Let's build a heatmap to see which texts are most similar.

In [ ]:
# ============================================================
# 📊 Cosine Similarity Heatmap
# ============================================================

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Build similarity matrix
n = len(texts)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = cosine_similarity(embeddings[i], embeddings[j])

# Short labels for the plot
labels = [t[:25] + "..." if len(t) > 25 else t for t in texts]

# Professional dark theme heatmap
fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")

im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha="right", color="white", fontsize=10)
ax.set_yticklabels(labels, color="white", fontsize=10)

# Add similarity values in each cell
for i in range(n):
    for j in range(n):
        color = "white" if sim_matrix[i][j] < 0.5 else "black"
        ax.text(j, i, f"{sim_matrix[i][j]:.2f}", ha="center", va="center", color=color, fontsize=9)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.ax.yaxis.set_tick_params(color="white")
cbar.ax.yaxis.set_ticklabels(cbar.ax.yaxis.get_ticklabels(), color="white")

ax.set_title("Cosine Similarity Between Text Embeddings", color="white", fontsize=16, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

# Highlight the most similar pairs
print("\n🔗 Most Similar Pairs (excluding self):")
pairs = []
for i in range(n):
    for j in range(i+1, n):
        pairs.append((sim_matrix[i][j], texts[i][:40], texts[j][:40]))
pairs.sort(reverse=True)
for score, t1, t2 in pairs[:4]:
    print(f"   {score:.3f}  {t1} ↔ {t2}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Notice that "stock market crashed" and "financial markets experienced a decline" have high similarity despite sharing almost no keywords. That's the power of <b>semantic</b> embeddings — they capture meaning, not just word overlap.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ ChromaDB Vector Store</h2>
</div>

A **vector store** is a database optimized for storing and searching embeddings. **ChromaDB** is a popular open-source option that runs locally (no server needed).

Think of it as: regular database searches by **exact match** → vector store searches by **meaning**.

In [ ]:
# ============================================================
# 🗄️ Set Up ChromaDB Vector Store
# ============================================================
import chromadb

# Create an in-memory client
chroma_client = chromadb.Client()

# Create a collection (like a table in a regular database)
collection = chroma_client.create_collection(
    name="gdp_facts",
    metadata={"description": "GDP and economic facts by country"}
)

# GDP data inspired by World Bank dataset
gdp_documents = [
    "The United States has the largest nominal GDP at approximately $25.5 trillion, driven primarily by the services sector including technology and finance.",
    "China's GDP is approximately $18 trillion, making it the second largest economy. Manufacturing and exports remain key growth drivers.",
    "Japan's GDP of $4.2 trillion makes it the third largest economy, with strengths in automotive manufacturing and technology.",
    "Germany leads the European economy with a GDP of $4.1 trillion, powered by engineering, automotive, and chemical industries.",
    "India's GDP of $3.4 trillion is growing rapidly at 6-7% annually, fueled by IT services, pharmaceuticals, and a young workforce.",
    "The United Kingdom has a GDP of $3.1 trillion with London serving as a global financial center.",
    "Brazil's GDP of $1.9 trillion is the largest in Latin America, with agriculture and mining as major sectors.",
    "Canada's GDP of $2.1 trillion benefits from natural resources, banking, and technology sectors.",
    "South Korea's GDP of $1.7 trillion is driven by electronics (Samsung, LG), automotive (Hyundai), and shipbuilding.",
    "Australia's GDP of $1.7 trillion relies on mining, agriculture, and a strong services sector.",
    "Global GDP growth slowed to 3.2% in 2023 due to inflation, rising interest rates, and geopolitical tensions.",
    "The GDP per capita of Luxembourg is the highest in the world at over $130,000, driven by financial services.",
]

# Generate embeddings and add to ChromaDB
emb_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=gdp_documents
)
doc_embeddings = [item.embedding for item in emb_response.data]

collection.add(
    documents=gdp_documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(gdp_documents))],
    metadatas=[{"source": "world_bank", "index": i} for i in range(len(gdp_documents))]
)

print(f"✅ Added {collection.count()} documents to ChromaDB")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Semantic Search: Query by Meaning</h2>
</div>

Now we can search by **meaning**, not just keywords. Watch how it finds relevant documents even when the query uses completely different words.

In [ ]:
# ============================================================
# 🔍 Semantic Search Queries
# ============================================================

def semantic_search(query, n_results=3):
    """Search ChromaDB by semantic meaning."""
    # Embed the query
    q_emb = client.embeddings.create(model="text-embedding-3-small", input=[query])
    q_vector = q_emb.data[0].embedding

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[q_vector],
        n_results=n_results
    )
    return results

# Test queries — note: these use different words than the documents!
queries = [
    "Which Asian countries have strong economies?",
    "What drives economic growth in emerging markets?",
    "Tell me about the richest country per person",
    "How did the world economy perform recently?",
]

for query in queries:
    results = semantic_search(query)
    search_output = []
    for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
        relevance = 1 - dist
        search_output.append({"rank": i + 1, "relevance": round(relevance, 3), "document": doc[:80] + "..."})
    pp(search_output, f"Search: {query}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Keyword Search vs Semantic Search</h2>
</div>

Let's compare the two approaches directly to see why semantic search is so powerful.

In [ ]:
# ============================================================
# ⚔️ Keyword Search vs Semantic Search
# ============================================================

def keyword_search(query, documents, n_results=3):
    """Simple keyword matching — count overlapping words."""
    query_words = set(query.lower().split())
    scores = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scores.append(overlap)
    # Return top n
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:n_results]
    return [(documents[i], s) for i, s in ranked]

# Test with a query that uses DIFFERENT words than the documents
test_query = "Which nation has the wealthiest citizens?"

print(f"🔍 Query: '{test_query}'\n")

# Keyword search
print("📝 KEYWORD SEARCH (word overlap):")
kw_results = keyword_search(test_query, gdp_documents)
for i, (doc, score) in enumerate(kw_results):
    print(f"   {i+1}. [score={score}] {doc[:70]}...")

# Semantic search
print(f"\n🧮 SEMANTIC SEARCH (meaning):")
sem_results = semantic_search(test_query)
for i, (doc, dist) in enumerate(zip(sem_results["documents"][0], sem_results["distances"][0])):
    print(f"   {i+1}. [sim={1-dist:.2f}] {doc[:70]}...")

print("\n💡 Semantic search found 'GDP per capita of Luxembourg' because it")
print("   understood 'wealthiest citizens' = 'highest GDP per capita' — even")
print("   though those words don't overlap at all!")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Keyword search fails when the query uses <b>different words</b> than the documents. Semantic search understands that "wealthiest citizens" ≈ "highest GDP per capita" because embeddings capture <b>meaning</b>, not just word overlap. This is why RAG systems use vector stores.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What does a text embedding represent?",
        "options": [
            "A compressed version of the original text",
            "A numerical vector that captures the semantic meaning of text",
            "A hash code for fast text lookup",
            "A translation of text into another language"
        ],
        "answer": 1,
        "explanation": "Embeddings are high-dimensional vectors (e.g., 1536 numbers) that represent the meaning of text. Similar meanings produce similar vectors, enabling semantic comparison."
    },
    {
        "q": "Why is semantic search better than keyword search for RAG?",
        "options": [
            "It's faster to compute",
            "It finds relevant documents even when different words are used to express the same meaning",
            "It returns more results",
            "It doesn't require an API key"
        ],
        "answer": 1,
        "explanation": "Semantic search matches by meaning, not exact words. A query about 'wealthiest citizens' can find documents about 'highest GDP per capita' — keyword search would miss this entirely."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add Your Own Documents and Query

Add 3 documents about a topic you choose (e.g., your favorite sport, a technology, a hobby) and query them semantically.

In [ ]:
# ============================================================
# 🔧 Exercise: Add Your Own Documents
# ============================================================
# Replace each ----- with the correct value

# Step 1: Create a new collection
my_collection = chroma_client.create_collection(name="-----")  # Name your collection

# Step 2: Define your documents (at least 3)
my_docs = [
    "-----",  # YOUR DOCUMENT 1
    "-----",  # YOUR DOCUMENT 2
    "-----",  # YOUR DOCUMENT 3
]

# Step 3: Generate embeddings
my_emb = client.embeddings.create(
    model="-----",                    # Which embedding model?
    input=my_docs
)
my_vectors = [item.embedding for item in my_emb.data]

# Step 4: Add to collection
my_collection.add(
    documents=my_docs,
    embeddings=my_vectors,
    ids=[f"my_{i}" for i in range(len(my_docs))]
)

# Step 5: Query!
my_query = "-----"  # YOUR QUERY — try using different words than your documents
q_emb = client.embeddings.create(model="text-embedding-3-small", input=[my_query])
results = my_collection.query(query_embeddings=[q_emb.data[0].embedding], n_results=2)

print(f"🔍 Query: {my_query}\n")
for i, doc in enumerate(results["documents"][0]):
    print(f"   {i+1}. {doc}")

**📝 Your Observations:** *(double-click to edit)*

1. Did the semantic search find the right documents for your query? _[Your answer]_

2. Search for "economic growth" vs "GDP increase" — do they return similar results? Why or why not? _[Your answer]_

3. Can you think of a query where semantic search might fail? (Hint: ambiguous words) _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these explorations:</p>
  <ol style="font-size: 14px;">
    <li>Add 10+ documents and test search quality — does it stay accurate?</li>
    <li>Try embedding sentences in <b>different languages</b> — does cosine similarity still work across languages?</li>
    <li>Compare <code>text-embedding-3-small</code> vs <code>text-embedding-3-large</code> — is there a quality difference?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've created embeddings, visualized similarity, and built a semantic search engine with ChromaDB.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Embeddings</b> convert text into numerical vectors that capture meaning</li>
      <li><b>Cosine similarity</b> measures how similar two embeddings are (0 to 1)</li>
      <li><b>ChromaDB</b> stores and searches embeddings efficiently</li>
      <li><b>Semantic search</b> finds relevant results even with different wording</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M05 Lab 2: Building a Full RAG Pipeline</p>
</div>